# `funder` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `high-cardinality-category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'funder'
feature_metadata = {'order': 3, 'name': 'funder', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'retain after conservative normalisation and fold-fitted rare grouping', 'finding': 'Effective missingness is about 7.5% and a small but material test share uses unseen funders.', 'decision': 'Keep blank and sentinel states distinct, normalise conservatively and handle unseen levels explicitly.', 'risk': 'Naive target encoding leaks and unrestricted fuzzy merging can combine different organisations.', 'sentinel_tokens': ['0', 'None', 'unknown', 'not known'], 'related': [{'feature': 'installer', 'reason': 'Funding and installation organisations are strongly associated but not duplicates.'}, {'feature': 'scheme_name', 'reason': 'A funder may repeatedly support named schemes.'}, {'feature': 'region', 'reason': 'Organisation activity is geographically concentrated.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for funder.


## Supported target evidence


In [2]:
sentinel_tokens = ['0', 'None', 'unknown', 'not known']
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
funder,,,,,
government of tanzania,9084,True,40.95,7.72,51.33
<missing/blank>,3635,True,54.50,12.02,33.48
danida,3114,True,55.01,5.11,39.88
hesawa,2202,True,42.51,10.54,46.96
rwssp,1374,True,58.59,7.93,33.48
world bank,1349,True,40.40,7.19,52.41
kkkt,1287,True,56.18,5.13,38.69
world vision,1246,True,59.63,10.51,29.86
unicef,1057,True,56.76,9.37,33.87


status_group,rows,non functional (%)
funder,,
fw,173,92.49
fini water,393,83.97
halmashauri ya wilaya sikonge,102,77.45
ru,105,76.19
finw,219,72.60
hsw,101,70.30
ir,123,66.67
ministry of water,590,64.24
w.b,170,62.35


## Observation

Effective missingness is about 7.5% and a small but material test share uses unseen funders.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Keep blank and sentinel states distinct, normalise conservatively and handle unseen levels explicitly.

**Risk to carry forward:** Naive target encoding leaks and unrestricted fuzzy merging can combine different organisations.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
funder,candidate,retain after conservative normalisation and fo...,Effective missingness is about 7.5% and a smal...,"Keep blank and sentinel states distinct, norma...",Naive target encoding leaks and unrestricted f...
